# Aula Prática 2 — Deep Learning (P1-10)

**Dataset:** Malaria Cell Images (NIH, ~27.5k imagens, 2 classes)  
**Objetivo:** Caracterização, divisão sem fugas, baseline CNN, curvas e diagnóstico.  
**Seed:** 42 em tudo  
**NOTA:** O conjunto de teste **NÃO** é avaliado nesta aula.

In [1]:
# ─── Imports e configuração global ───────────────────────────────────────
import os, re, hashlib, shutil, pathlib, warnings, time
import numpy as np
import pandas as pd
from PIL import Image
from collections import Counter
import matplotlib
matplotlib.use('Agg')  # backend não-interativo para garantir que funciona headless
import matplotlib.pyplot as plt

# Seed global
SEED = 42
np.random.seed(SEED)

# Keras 3 + PyTorch backend
os.environ['KERAS_BACKEND'] = 'torch'
import torch
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

import keras
keras.utils.set_random_seed(SEED)

# Caminhos
ROOT = pathlib.Path(os.path.abspath(os.path.join(os.getcwd(), '..')))
DATA_RAW = ROOT / 'data' / 'raw'
AULA2 = ROOT / 'aula2'
SPLITS_D1 = ROOT / 'splits'

# Verificação
print(f'ROOT:     {ROOT}')
print(f'DATA_RAW: {DATA_RAW} (existe: {DATA_RAW.exists()})')
print(f'Keras:    {keras.__version__}')
print(f'PyTorch:  {torch.__version__}')
print(f'GPU:      {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Sem GPU"}')

ROOT:     C:\Users\vilar\Desktop\Mestrado\deeeplearning
DATA_RAW: C:\Users\vilar\Desktop\Mestrado\deeeplearning\data\raw (existe: True)
Keras:    3.15.1
PyTorch:  2.11.0+cu128
GPU:      NVIDIA GeForce RTX 5060 Laptop GPU


---
## 2.1 — Caracterização do Dataset

In [2]:
# ─── 2.1a  Listar todas as imagens ──────────────────────────────────────
# Percorre cell_images/ dentro de data/raw/ e recolhe jpg/jpeg/png

IMG_DIR = DATA_RAW / 'cell_images'
VALID_EXT = {'.jpg', '.jpeg', '.png'}

records = []
for class_dir in sorted(IMG_DIR.iterdir()):
    if not class_dir.is_dir():
        continue
    # Ignorar a pasta cell_images/cell_images (aninhada do zip)
    if class_dir.name == 'cell_images':
        continue
    label = class_dir.name
    for f in sorted(class_dir.iterdir()):
        if f.suffix.lower() in VALID_EXT:
            records.append({'path': str(f), 'label': label, 'filename': f.name})

df = pd.DataFrame(records)
print(f'Total de imagens encontradas: {len(df)}')
df.head()

Total de imagens encontradas: 27558


,path,label,filename
0,C:\Users\vilar\Desktop\Mestrado\deeeplearning\...,Parasitized,C100P61ThinF_IMG_20150918_144104_cell_162.png
1,C:\Users\vilar\Desktop\Mestrado\deeeplearning\...,Parasitized,C100P61ThinF_IMG_20150918_144104_cell_163.png
2,C:\Users\vilar\Desktop\Mestrado\deeeplearning\...,Parasitized,C100P61ThinF_IMG_20150918_144104_cell_164.png
3,C:\Users\vilar\Desktop\Mestrado\deeeplearning\...,Parasitized,C100P61ThinF_IMG_20150918_144104_cell_165.png
4,C:\Users\vilar\Desktop\Mestrado\deeeplearning\...,Parasitized,C100P61ThinF_IMG_20150918_144104_cell_166.png


In [3]:
# ─── 2.1b  Extrair ID de doente (grupo) com a regex do D1 ──────────────
# Regex: ^(C\d+(?:P\d+)?)  — extrai o código do paciente/lâmina

GROUP_REGEX = r'^(C\d+(?:P\d+)?)'

def extract_group(filename):
    m = re.match(GROUP_REGEX, filename)
    return m.group(1) if m else 'UNKNOWN'

df['grupo'] = df['filename'].apply(extract_group)

n_unknown = (df['grupo'] == 'UNKNOWN').sum()
print(f'Imagens sem grupo identificado: {n_unknown}')
print(f'Grupos únicos: {df["grupo"].nunique()}')
df[['filename', 'label', 'grupo']].head(10)

Imagens sem grupo identificado: 0
Grupos únicos: 200


,filename,label,grupo
0,C100P61ThinF_IMG_20150918_144104_cell_162.png,Parasitized,C100P61
1,C100P61ThinF_IMG_20150918_144104_cell_163.png,Parasitized,C100P61
2,C100P61ThinF_IMG_20150918_144104_cell_164.png,Parasitized,C100P61
3,C100P61ThinF_IMG_20150918_144104_cell_165.png,Parasitized,C100P61
4,C100P61ThinF_IMG_20150918_144104_cell_166.png,Parasitized,C100P61
5,C100P61ThinF_IMG_20150918_144104_cell_167.png,Parasitized,C100P61
6,C100P61ThinF_IMG_20150918_144104_cell_168.png,Parasitized,C100P61
7,C100P61ThinF_IMG_20150918_144104_cell_169.png,Parasitized,C100P61
8,C100P61ThinF_IMG_20150918_144104_cell_170.png,Parasitized,C100P61
9,C100P61ThinF_IMG_20150918_144104_cell_171.png,Parasitized,C100P61


In [4]:
# ─── 2.1c  Value counts por classe ─────────────────────────────────────
vc_abs = df['label'].value_counts()
vc_pct = df['label'].value_counts(normalize=True) * 100

dist_df = pd.DataFrame({'N.º imagens': vc_abs, '% do total': vc_pct.round(2)})
dist_df.index.name = 'Classe'
print('=== Distribuição por classe ===')
display(dist_df)
print(f'\nTotal: {len(df)} imagens')

=== Distribuição por classe ===


,N.º imagens,% do total
Classe,,
Parasitized,13779,50.0
Uninfected,13779,50.0



Total: 27558 imagens


In [5]:
# ─── 2.1d  Dimensão típica por classe + deteção de imagens corrompidas ─
widths, heights, corrupted_paths = [], [], []

for idx, row in df.iterrows():
    try:
        with Image.open(row['path']) as img:
            img.verify()  # Verifica integridade
        # Re-open para obter dimensões (verify() invalida o objeto)
        with Image.open(row['path']) as img:
            w, h = img.size
            widths.append(w)
            heights.append(h)
    except Exception as e:
        corrupted_paths.append((row['path'], str(e)))
        widths.append(None)
        heights.append(None)

df['width'] = widths
df['height'] = heights

print(f'Imagens corrompidas: {len(corrupted_paths)}')
if corrupted_paths:
    for p, e in corrupted_paths:
        print(f'  {p}: {e}')

# Dimensão típica por classe (mediana, min, max)
dim_stats = df.dropna(subset=['width', 'height']).groupby('label').agg(
    mediana_w=('width', 'median'),
    mediana_h=('height', 'median'),
    min_w=('width', 'min'),
    max_w=('width', 'max'),
    min_h=('height', 'min'),
    max_h=('height', 'max'),
).astype(int)

dim_stats['Dimensão típica'] = dim_stats.apply(
    lambda r: f"{r['mediana_w']}×{r['mediana_h']}", axis=1
)
dim_stats['Intervalo'] = dim_stats.apply(
    lambda r: f"({r['min_w']}–{r['max_w']}) × ({r['min_h']}–{r['max_h']})", axis=1
)

print('\n=== Dimensões por classe ===')
display(dim_stats[['Dimensão típica', 'Intervalo']])

Imagens corrompidas: 0

=== Dimensões por classe ===


,Dimensão típica,Intervalo
label,,
Parasitized,133×133,(46–394) × (40–385)
Uninfected,130×130,(49–247) × (49–235)


In [6]:
# ─── 2.1e  Deteção de duplicados por MD5 ───────────────────────────────

def file_md5(filepath):
    h = hashlib.md5()
    with open(filepath, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()

print('A calcular MD5 de todas as imagens...')
df['md5'] = df['path'].apply(file_md5)

# Duplicados: mesmo md5
dup_mask = df.duplicated(subset='md5', keep=False)
n_dup_groups = df[dup_mask]['md5'].nunique()
n_dup_total = dup_mask.sum()

print(f'\nImagens com hash duplicado: {n_dup_total} (em {n_dup_groups} grupos de duplicados)')

if n_dup_total > 0:
    # Mostrar alguns exemplos
    dup_examples = df[dup_mask].sort_values('md5').head(10)
    display(dup_examples[['filename', 'label', 'grupo', 'md5']])
    
    # Duplicados entre classes?
    cross_class = df[dup_mask].groupby('md5')['label'].nunique()
    cross_class_dup = (cross_class > 1).sum()
    print(f'Duplicados entre classes diferentes: {cross_class_dup}')

A calcular MD5 de todas as imagens...



Imagens com hash duplicado: 0 (em 0 grupos de duplicados)


In [7]:
# ─── 2.1f  Tabela final de caracterização ──────────────────────────────

# Construir observações com base nos dados encontrados
def build_obs(label):
    obs_parts = []
    
    # Corrompidas nesta classe?
    n_corr = len([p for p, _ in corrupted_paths if label in p])
    if n_corr > 0:
        obs_parts.append(f'{n_corr} imagem(ns) corrompida(s)')
    
    # Duplicados nesta classe
    class_dups = df[(df['label'] == label) & dup_mask]
    if len(class_dups) > 0:
        obs_parts.append(f'{len(class_dups)} imgs com hash duplicado')
    
    # Variação de dimensão
    sub = df[df['label'] == label].dropna(subset=['width'])
    w_range = sub['width'].max() - sub['width'].min()
    h_range = sub['height'].max() - sub['height'].min()
    if w_range > 100 or h_range > 100:
        obs_parts.append(f'dimensões variáveis (range {w_range}×{h_range})')
    else:
        obs_parts.append('dimensões relativamente homogéneas')
    
    # Imagens sem grupo
    n_unk = ((df['label'] == label) & (df['grupo'] == 'UNKNOWN')).sum()
    if n_unk > 0:
        obs_parts.append(f'{n_unk} sem ID de doente')
    
    if not obs_parts:
        obs_parts.append('sem problemas detetados')
    
    return '; '.join(obs_parts)

summary_rows = []
for label in sorted(df['label'].unique()):
    sub = df[df['label'] == label]
    sub_clean = sub.dropna(subset=['width'])
    summary_rows.append({
        'Classe': label,
        'N.º imagens': len(sub),
        '%': f"{len(sub)/len(df)*100:.1f}%",
        'Dimensão típica': f"{int(sub_clean['width'].median())}×{int(sub_clean['height'].median())}",
        'Observações': build_obs(label),
    })

summary_table = pd.DataFrame(summary_rows)
print('=== Tabela 2.1 — Caracterização do Dataset ===')
display(summary_table)

# Guardar para uso posterior
summary_table.to_csv(AULA2 / 'tabela_21_caracterizacao.csv', index=False)
print(f'\nTabela guardada em aula2/tabela_21_caracterizacao.csv')

=== Tabela 2.1 — Caracterização do Dataset ===


,Classe,N.º imagens,%,Dimensão típica,Observações
0,Parasitized,13779,50.0%,133×133,dimensões variáveis (range 348×345)
1,Uninfected,13779,50.0%,130×130,dimensões variáveis (range 198×186)



Tabela guardada em aula2/tabela_21_caracterizacao.csv


---
## 2.2 — Divisão sem fugas (Group-Aware Split)

### Escolha de abordagem

**Decisão:** Reaproveitar a divisão do D1 (`splits/train.csv`, `val.csv`, `test.csv`).

**Justificação:**
- A divisão do D1 já foi feita com `GroupShuffleSplit` (seed 42), respeitando a integridade de grupos (doentes).
- Reaproveitar garante consistência entre aulas e evita resultados incomparáveis.
- Vamos validar que a divisão cumpre todos os requisitos (sem fugas de grupo, sem fugas por md5, estratificação).

In [8]:
# ─── 2.2a  Carregar splits do D1 e construir splits.csv ────────────────

train_d1 = pd.read_csv(SPLITS_D1 / 'train.csv')
val_d1   = pd.read_csv(SPLITS_D1 / 'val.csv')
test_d1  = pd.read_csv(SPLITS_D1 / 'test.csv')

train_d1['split'] = 'train'
val_d1['split']   = 'val'
test_d1['split']  = 'test'

splits_d1 = pd.concat([train_d1, val_d1, test_d1], ignore_index=True)
splits_d1.rename(columns={'relative_path': 'rel_path', 'class': 'label', 'id_grupo': 'grupo'}, inplace=True)

# Construir path absoluto para fazer join com o nosso df
splits_d1['path'] = splits_d1['rel_path'].apply(lambda p: str(DATA_RAW / p))

print(f'Total D1 splits: {len(splits_d1)}')
print(f'Total nosso df:  {len(df)}')

# Join com md5 do nosso df
splits_merged = splits_d1.merge(df[['path', 'md5']], on='path', how='left')

# Verificar se há NaN no md5 (imagens no D1 que não encontrámos)
n_missing = splits_merged['md5'].isna().sum()
print(f'Imagens no D1 sem correspondência no nosso scan: {n_missing}')

# Guardar splits.csv
splits_final = splits_merged[['path', 'label', 'grupo', 'md5', 'split']].copy()
splits_final.to_csv(AULA2 / 'splits.csv', index=False)
print(f'\nGuardado aula2/splits.csv com {len(splits_final)} linhas')
splits_final.head()

Total D1 splits: 27558
Total nosso df:  27558
Imagens no D1 sem correspondência no nosso scan: 0



Guardado aula2/splits.csv com 27558 linhas


,path,label,grupo,md5,split
0,C:\Users\vilar\Desktop\Mestrado\deeeplearning\...,Parasitized,C100P61,6b60535df837d603b2f591d154940f1a,train
1,C:\Users\vilar\Desktop\Mestrado\deeeplearning\...,Parasitized,C100P61,142cdfcc8f5d2afcdd327e4886a4fbf8,train
2,C:\Users\vilar\Desktop\Mestrado\deeeplearning\...,Parasitized,C100P61,2db29c733097f1736cdba4055e1549d6,train
3,C:\Users\vilar\Desktop\Mestrado\deeeplearning\...,Parasitized,C100P61,2fafd2198f16e9d264cd0426f164e5a6,train
4,C:\Users\vilar\Desktop\Mestrado\deeeplearning\...,Parasitized,C100P61,6da8225a1d75edded5ecb514e63f1c92,train


In [9]:
# ─── 2.2b  Verificações de integridade ─────────────────────────────────

train_s = splits_final[splits_final['split'] == 'train']
val_s   = splits_final[splits_final['split'] == 'val']
test_s  = splits_final[splits_final['split'] == 'test']

print('=== Verificação 1: Interseção de MD5 entre splits ===')
md5_train = set(train_s['md5'].dropna())
md5_val   = set(val_s['md5'].dropna())
md5_test  = set(test_s['md5'].dropna())

inter_tv = md5_train & md5_val
inter_tt = md5_train & md5_test
inter_vt = md5_val & md5_test

print(f'  train ∩ val  (md5): {len(inter_tv)}')
print(f'  train ∩ test (md5): {len(inter_tt)}')
print(f'  val   ∩ test (md5): {len(inter_vt)}')

if len(inter_tv) + len(inter_tt) + len(inter_vt) == 0:
    print('  ✅ Sem fugas por MD5!')
else:
    print('  ⚠️ AVISO: Existem hashes duplicados entre splits!')
    if inter_tv:
        print(f'    MD5s em train∩val: {list(inter_tv)[:5]}')
    if inter_tt:
        print(f'    MD5s em train∩test: {list(inter_tt)[:5]}')
    if inter_vt:
        print(f'    MD5s em val∩test: {list(inter_vt)[:5]}')

=== Verificação 1: Interseção de MD5 entre splits ===
  train ∩ val  (md5): 0
  train ∩ test (md5): 0
  val   ∩ test (md5): 0
  ✅ Sem fugas por MD5!


In [10]:
# ─── 2.2c  Verificação de interseção de GRUPOS ─────────────────────────

print('=== Verificação 2: Interseção de GRUPOS entre splits ===')
grp_train = set(train_s['grupo'])
grp_val   = set(val_s['grupo'])
grp_test  = set(test_s['grupo'])

inter_g_tv = grp_train & grp_val
inter_g_tt = grp_train & grp_test
inter_g_vt = grp_val & grp_test

print(f'  train ∩ val  (grupo): {len(inter_g_tv)}')
print(f'  train ∩ test (grupo): {len(inter_g_tt)}')
print(f'  val   ∩ test (grupo): {len(inter_g_vt)}')

if len(inter_g_tv) + len(inter_g_tt) + len(inter_g_vt) == 0:
    print('  ✅ Sem fugas de grupo!')
else:
    print('  ⚠️ AVISO: Existem grupos partilhados entre splits!')
    if inter_g_tv:
        print(f'    Grupos em train∩val: {sorted(inter_g_tv)[:10]}')
    if inter_g_tt:
        print(f'    Grupos em train∩test: {sorted(inter_g_tt)[:10]}')
    if inter_g_vt:
        print(f'    Grupos em val∩test: {sorted(inter_g_vt)[:10]}')

=== Verificação 2: Interseção de GRUPOS entre splits ===
  train ∩ val  (grupo): 0
  train ∩ test (grupo): 0
  val   ∩ test (grupo): 0
  ✅ Sem fugas de grupo!


In [11]:
# ─── 2.2d  Distribuição por classe em cada split ───────────────────────

print('=== Verificação 3: Distribuição por classe em cada split ===')
for split_name in ['train', 'val', 'test']:
    sub = splits_final[splits_final['split'] == split_name]
    pct = sub['label'].value_counts(normalize=True) * 100
    print(f'\n{split_name.upper()} ({len(sub)} imagens):')
    for cls in sorted(pct.index):
        print(f'  {cls}: {pct[cls]:.1f}%')

# Verificar desvio máximo
all_pcts = {}
for split_name in ['train', 'val', 'test']:
    sub = splits_final[splits_final['split'] == split_name]
    pct = sub['label'].value_counts(normalize=True) * 100
    all_pcts[split_name] = pct

pct_df = pd.DataFrame(all_pcts).T
max_diff = pct_df.max() - pct_df.min()
print(f'\nDesvio máximo entre splits por classe:')
for cls in max_diff.index:
    diff = max_diff[cls]
    status = '✅' if diff < 2.0 else '⚠️'
    print(f'  {cls}: {diff:.2f} pp {status}')

display(pct_df.round(2))

=== Verificação 3: Distribuição por classe em cada split ===

TRAIN (19380 imagens):
  Parasitized: 50.1%
  Uninfected: 49.9%

VAL (4107 imagens):
  Parasitized: 50.0%
  Uninfected: 50.0%

TEST (4071 imagens):
  Parasitized: 49.4%
  Uninfected: 50.6%

Desvio máximo entre splits por classe:
  Parasitized: 0.75 pp ✅
  Uninfected: 0.75 pp ✅


label,Parasitized,Uninfected
train,50.12,49.88
val,50.04,49.96
test,49.37,50.63


---
## 2.3 — Baseline CNN

In [12]:
# ─── 2.3a  Criar estrutura de pastas com cópias/symlinks ───────────────
SPLIT_DIR = AULA2 / 'split'

# Limpar se já existe
if SPLIT_DIR.exists():
    shutil.rmtree(SPLIT_DIR)

created_count = {'train': 0, 'val': 0, 'test': 0}

for _, row in splits_final.iterrows():
    src = pathlib.Path(row['path'])
    dst_dir = SPLIT_DIR / row['split'] / row['label']
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = dst_dir / src.name
    
    if not dst.exists():
        try:
            # Tentar symlink primeiro
            os.symlink(src, dst)
        except OSError:
            # Fallback para cópia no Windows sem permissões de symlink
            shutil.copy2(src, dst)
        created_count[row['split']] += 1

print('Ficheiros criados por split:')
for k, v in created_count.items():
    print(f'  {k}: {v}')

# Adicionar aula2/split/ ao .gitignore
gitignore_path = ROOT / '.gitignore'
gitignore_entry = 'aula2/split/'
with open(gitignore_path, 'r') as f:
    content = f.read()
if gitignore_entry not in content:
    with open(gitignore_path, 'a') as f:
        f.write(f'\n# Aula 2 — pastas de imagens (cópias/symlinks)\n{gitignore_entry}\n')
    print(f'\nAdicionado "{gitignore_entry}" ao .gitignore')
else:
    print(f'\n"{gitignore_entry}" já está no .gitignore')

Ficheiros criados por split:
  train: 19380
  val: 4107
  test: 4071

"aula2/split/" já está no .gitignore


In [13]:
# ─── 2.3b  Carregar datasets com Keras ─────────────────────────────────
IMG_SIZE = (180, 180)
BATCH_SIZE = 32

train_ds = keras.utils.image_dataset_from_directory(
    str(SPLIT_DIR / 'train'),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=True,
    seed=SEED,
)

val_ds = keras.utils.image_dataset_from_directory(
    str(SPLIT_DIR / 'val'),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=False,
)

class_names = train_ds.class_names
num_classes = len(class_names)
print(f'Classes: {class_names}')
print(f'Número de classes: {num_classes}')

Found 19380 files belonging to 2 classes.


Found 4107 files belonging to 2 classes.


Classes: ['Parasitized', 'Uninfected']
Número de classes: 2


In [14]:
# ─── 2.3c  Definir a CNN baseline (arquitetura exata do enunciado) ─────
from keras import layers, models

def build_baseline_cnn(num_classes, input_shape=(180, 180, 3)):
    inputs = keras.Input(shape=input_shape)
    x = layers.Rescaling(1.0 / 255)(inputs)
    
    # 4 blocos Conv2D + MaxPooling2D com filtros [32, 64, 128, 256]
    for filters in [32, 64, 128, 256]:
        x = layers.Conv2D(filters, 3, activation='relu')(x)
        x = layers.MaxPooling2D(2)(x)
    
    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = models.Model(inputs, outputs)
    return model

model = build_baseline_cnn(num_classes)

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 180, 180, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 180, 180, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 178, 178, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 89, 89, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 87, 87, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 43, 43, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 41, 41, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 20, 20, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 18, 18, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 9, 9, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │           514 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 388,930 (1.48 MB)

 Trainable params: 388,930 (1.48 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
# ─── 2.3d  Confirmação manual do nº de parâmetros ──────────────────────
print('=== Verificação manual dos parâmetros das duas primeiras convoluções ===\n')

# Conv2D_1: input 3 canais, 32 filtros, kernel 3×3
# Parâmetros = (kernel_h × kernel_w × input_channels + 1_bias) × n_filters
conv1_params = (3 * 3 * 3 + 1) * 32
print(f'Conv2D #1 (3→32, kernel 3×3):')
print(f'  Cálculo: (3 × 3 × 3 + 1) × 32 = ({3*3*3} + 1) × 32 = {3*3*3+1} × 32 = {conv1_params}')

# Conv2D_2: input 32 canais, 64 filtros, kernel 3×3
conv2_params = (3 * 3 * 32 + 1) * 64
print(f'\nConv2D #2 (32→64, kernel 3×3):')
print(f'  Cálculo: (3 × 3 × 32 + 1) × 64 = ({3*3*32} + 1) × 64 = {3*3*32+1} × 64 = {conv2_params}')

# Comparar com o summary
layer_configs = []
for layer in model.layers:
    if 'conv2d' in layer.name:
        layer_configs.append((layer.name, layer.count_params()))

print(f'\n--- Comparação com model.summary() ---')
print(f'Conv2D #1: cálculo manual = {conv1_params}, summary = {layer_configs[0][1]} → {"✅ OK" if conv1_params == layer_configs[0][1] else "❌ DIFERENTE"}')
print(f'Conv2D #2: cálculo manual = {conv2_params}, summary = {layer_configs[1][1]} → {"✅ OK" if conv2_params == layer_configs[1][1] else "❌ DIFERENTE"}')

=== Verificação manual dos parâmetros das duas primeiras convoluções ===

Conv2D #1 (3→32, kernel 3×3):
  Cálculo: (3 × 3 × 3 + 1) × 32 = (27 + 1) × 32 = 28 × 32 = 896

Conv2D #2 (32→64, kernel 3×3):
  Cálculo: (3 × 3 × 32 + 1) × 64 = (288 + 1) × 64 = 289 × 64 = 18496

--- Comparação com model.summary() ---
Conv2D #1: cálculo manual = 896, summary = 896 → ✅ OK
Conv2D #2: cálculo manual = 18496, summary = 18496 → ✅ OK


In [16]:
# ─── 2.3e  Treinar o modelo (30 épocas, sem callbacks) ─────────────────
EPOCHS = 30

print(f'Início do treino: {time.strftime("%H:%M:%S")}')
t0 = time.time()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    verbose=1,
)

t1 = time.time()
train_time = t1 - t0
print(f'\nFim do treino: {time.strftime("%H:%M:%S")}')
print(f'Tempo total de treino: {train_time:.1f}s ({train_time/60:.1f} min)')

Início do treino: 13:20:43


C:\Users\vilar\Desktop\Mestrado\deeeplearning\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


Epoch 1/30


  1/606 ━━━━━━━━━━━━━━━━━━━━ 11:59 1s/step - accuracy: 0.4688 - loss: 0.6934

  3/606 ━━━━━━━━━━━━━━━━━━━━ 23s 39ms/step - accuracy: 0.5104 - loss: 0.6968

...

602/606 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.9884 - loss: 0.0347

604/606 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.9884 - loss: 0.0348

606/606 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.9884 - loss: 0.0348

606/606 ━━━━━━━━━━━━━━━━━━━━ 30s 50ms/step - accuracy: 0.9884 - loss: 0.0348 - val_accuracy: 0.9364 - val_loss: 0.3411



Fim do treino: 13:34:50
Tempo total de treino: 847.0s (14.1 min)


In [17]:
# ─── 2.3f  Classificador maioritário ───────────────────────────────────

# Contar labels no val
val_label_counts = val_s['label'].value_counts()
majority_class = val_label_counts.idxmax()
majority_acc = val_label_counts.max() / len(val_s)

# Acurácia de validação final do modelo
final_val_acc = history.history['val_accuracy'][-1]

print(f'=== Classificador Maioritário ===')
print(f'Classe maioritária no val: {majority_class} ({val_label_counts.max()}/{len(val_s)})')
print(f'Acurácia do classificador maioritário: {majority_acc:.4f} ({majority_acc*100:.2f}%)')
print(f'\nAcurácia de validação final (época 30): {final_val_acc:.4f} ({final_val_acc*100:.2f}%)')
print(f'Diferença: {(final_val_acc - majority_acc)*100:+.2f} pp')

if final_val_acc > majority_acc:
    print('\n✅ O modelo supera o classificador maioritário.')
else:
    print('\n⚠️ O modelo NÃO supera o classificador maioritário!')

=== Classificador Maioritário ===
Classe maioritária no val: Parasitized (2055/4107)
Acurácia do classificador maioritário: 0.5004 (50.04%)

Acurácia de validação final (época 30): 0.9364 (93.64%)
Diferença: +43.61 pp

✅ O modelo supera o classificador maioritário.


---
## 2.4 — Curvas e Diagnóstico

In [18]:
# ─── 2.4a  Guardar history em CSV ──────────────────────────────────────
hist_df = pd.DataFrame(history.history)
hist_df.index += 1  # épocas 1-indexed
hist_df.index.name = 'epoch'
hist_df.to_csv(AULA2 / 'history_baseline.csv')
print(f'History guardado em aula2/history_baseline.csv ({len(hist_df)} épocas)')
hist_df.head()

History guardado em aula2/history_baseline.csv (30 épocas)


,accuracy,loss,val_accuracy,val_loss
epoch,,,,
1,0.846594,0.417272,0.940833,0.284739
2,0.917183,0.320591,0.943268,0.404837
3,0.939525,0.191080,0.953737,0.175435
4,0.945304,0.166640,0.943755,0.228984
5,0.947988,0.162414,0.952277,0.190497


In [19]:
# ─── 2.4b  Plot de loss e accuracy ─────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, EPOCHS + 1)

# Loss
ax1.plot(epochs_range, hist_df['loss'], 'b-o', markersize=3, label='Treino')
ax1.plot(epochs_range, hist_df['val_loss'], 'r-o', markersize=3, label='Validação')
ax1.set_xlabel('Época')
ax1.set_ylabel('Loss')
ax1.set_title('Loss por Época')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy
ax2.plot(epochs_range, hist_df['accuracy'], 'b-o', markersize=3, label='Treino')
ax2.plot(epochs_range, hist_df['val_accuracy'], 'r-o', markersize=3, label='Validação')
ax2.set_xlabel('Época')
ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy por Época')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Baseline CNN — Curvas de Aprendizagem', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(AULA2 / 'curvas_baseline.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico guardado em aula2/curvas_baseline.png')

Gráfico guardado em aula2/curvas_baseline.png


C:\Users\vilar\AppData\Local\Temp\ipykernel_26088\2295760125.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [20]:
# ─── 2.4c  Tabela de diagnóstico ───────────────────────────────────────

# Melhor época (menor val_loss)
best_epoch = int(hist_df['val_loss'].idxmin())
best_val_loss = hist_df.loc[best_epoch, 'val_loss']
best_train_acc = hist_df.loc[best_epoch, 'accuracy']
best_val_acc = hist_df.loc[best_epoch, 'val_accuracy']
best_train_loss = hist_df.loc[best_epoch, 'loss']

# Diagnóstico baseado nos números
gap_acc = best_train_acc - best_val_acc
gap_loss = best_val_loss - best_train_loss

if best_val_acc < majority_acc + 0.05:
    diagnostico = 'Underfitting'
    justificacao = (f'A acurácia de validação ({best_val_acc:.4f}) está próxima '
                    f'do classificador maioritário ({majority_acc:.4f}), '
                    f'indicando que o modelo não consegue aprender padrões úteis.')
elif gap_acc > 0.10:
    diagnostico = 'Overfitting'
    justificacao = (f'Gap treino-val de {gap_acc:.4f} em accuracy e {gap_loss:.4f} em loss. '
                    f'O modelo memoriza o treino (acc={best_train_acc:.4f}) '
                    f'mas generaliza mal (val_acc={best_val_acc:.4f}).')
elif gap_acc > 0.05:
    diagnostico = 'Overfitting ligeiro'
    justificacao = (f'Gap treino-val de {gap_acc:.4f} em accuracy. '
                    f'Acc treino={best_train_acc:.4f}, val={best_val_acc:.4f}. '
                    f'Há sinais de sobreajuste mas o modelo ainda generaliza razoavelmente.')
else:
    diagnostico = 'Adequado'
    justificacao = (f'Gap treino-val pequeno ({gap_acc:.4f} em accuracy). '
                    f'Acc treino={best_train_acc:.4f}, val={best_val_acc:.4f}. '
                    f'O modelo generaliza bem.')

print('=== Tabela 2.4 — Diagnóstico ===')
print(f'Época com menor val_loss:     {best_epoch}')
print(f'Val loss (nessa época):       {best_val_loss:.4f}')
print(f'Acc treino (nessa época):     {best_train_acc:.4f} ({best_train_acc*100:.2f}%)')
print(f'Acc val (nessa época):        {best_val_acc:.4f} ({best_val_acc*100:.2f}%)')
print(f'Classificador maioritário:    {majority_acc:.4f} ({majority_acc*100:.2f}%)')
print(f'Diagnóstico:                  {diagnostico}')
print(f'Justificação:                 {justificacao}')

# Última época para comparação
print(f'\n--- Última época (30) ---')
print(f'Acc treino: {hist_df.loc[30, "accuracy"]:.4f}')
print(f'Acc val:    {hist_df.loc[30, "val_accuracy"]:.4f}')
print(f'Loss treino: {hist_df.loc[30, "loss"]:.4f}')
print(f'Loss val:    {hist_df.loc[30, "val_loss"]:.4f}')

=== Tabela 2.4 — Diagnóstico ===
Época com menor val_loss:     7
Val loss (nessa época):       0.1387
Acc treino (nessa época):     0.9491 (94.91%)
Acc val (nessa época):        0.9627 (96.27%)
Classificador maioritário:    0.5004 (50.04%)
Diagnóstico:                  Adequado
Justificação:                 Gap treino-val pequeno (-0.0136 em accuracy). Acc treino=0.9491, val=0.9627. O modelo generaliza bem.

--- Última época (30) ---
Acc treino: 0.9884
Acc val:    0.9364
Loss treino: 0.0348
Loss val:    0.3411


In [21]:
# ─── 2.4d  Gerar RESULTADOS.md ────────────────────────────────────────

resultados_md = f"""# Aula Prática 2 — Resultados

## 2.1 — Caracterização do Dataset

| Classe | N.º imagens | % | Dimensão típica | Observações |
|--------|-------------|---|-----------------|-------------|
"""

for _, row in summary_table.iterrows():
    resultados_md += f"| {row['Classe']} | {row['N.º imagens']} | {row['%']} | {row['Dimensão típica']} | {row['Observações']} |\n"

resultados_md += f"""
**Total:** {len(df)} imagens  
**Grupos únicos:** {df['grupo'].nunique()}  
**Imagens corrompidas:** {len(corrupted_paths)}  
**Duplicados (md5):** {n_dup_total} imagens em {n_dup_groups} grupos

## 2.2 — Divisão sem fugas

**Método:** Reutilização dos splits do D1 (GroupShuffleSplit, seed 42).  
**Justificação:** Garante consistência com o trabalho anterior e respeita a integridade de grupos.

| Verificação | Resultado |
|------------|----------|
| Interseção MD5 train∩val | {len(inter_tv)} |
| Interseção MD5 train∩test | {len(inter_tt)} |
| Interseção MD5 val∩test | {len(inter_vt)} |
| Interseção grupo train∩val | {len(inter_g_tv)} |
| Interseção grupo train∩test | {len(inter_g_tt)} |
| Interseção grupo val∩test | {len(inter_g_vt)} |

| Split | N | Parasitized % | Uninfected % |
|-------|---|--------------|-------------|
"""

for split_name in ['train', 'val', 'test']:
    sub = splits_final[splits_final['split'] == split_name]
    pct = sub['label'].value_counts(normalize=True) * 100
    p_pct = pct.get('Parasitized', 0)
    u_pct = pct.get('Uninfected', 0)
    resultados_md += f"| {split_name} | {len(sub)} | {p_pct:.1f}% | {u_pct:.1f}% |\n"

resultados_md += f"""
## 2.3 — Baseline CNN

**Arquitetura:** Input(180,180,3) → Rescaling(1/255) → 4×[Conv2D(f,3,relu)+MaxPool(2)] com f∈[32,64,128,256] → GAP → Dense({num_classes}, softmax)  
**Optimizer:** Adam  
**Loss:** sparse_categorical_crossentropy  
**Épocas:** {EPOCHS} (sem EarlyStopping)  
**Tempo de treino:** {train_time:.1f}s ({train_time/60:.1f} min)

Verificação de parâmetros:
- Conv2D #1: (3×3×3+1)×32 = {conv1_params} ({'✅' if conv1_params == layer_configs[0][1] else '❌'})
- Conv2D #2: (3×3×32+1)×64 = {conv2_params} ({'✅' if conv2_params == layer_configs[1][1] else '❌'})

## 2.4 — Diagnóstico

| Métrica | Valor |
|---------|-------|
| Época com menor val_loss | {best_epoch} |
| Acc treino (nessa época) | {best_train_acc:.4f} ({best_train_acc*100:.2f}%) |
| Acc val (nessa época) | {best_val_acc:.4f} ({best_val_acc*100:.2f}%) |
| Classificador maioritário | {majority_acc:.4f} ({majority_acc*100:.2f}%) |
| Diagnóstico | {diagnostico} |

**Justificação:** {justificacao}

## Notas / O que correu mal

"""

# Notas baseadas na execução
notas = []
if len(corrupted_paths) > 0:
    notas.append(f'- Encontradas {len(corrupted_paths)} imagem(ns) corrompida(s)')
if n_dup_total > 0:
    notas.append(f'- Encontrados {n_dup_total} ficheiros com hash MD5 duplicado ({n_dup_groups} grupos)')
if len(inter_tv) + len(inter_tt) + len(inter_vt) > 0:
    notas.append('- ⚠️ Existem hashes duplicados entre splits (possível fuga de dados)')
if len(inter_g_tv) + len(inter_g_tt) + len(inter_g_vt) > 0:
    notas.append('- ⚠️ Existem grupos partilhados entre splits (fuga de grupo)')
if n_missing > 0:
    notas.append(f'- {n_missing} imagens nos splits D1 não encontradas no scan (possível limpeza/remoção)')
if not notas:
    notas.append('- Nenhum problema encontrado')

resultados_md += '\n'.join(notas) + '\n'

with open(AULA2 / 'RESULTADOS.md', 'w', encoding='utf-8') as f:
    f.write(resultados_md)

print('RESULTADOS.md criado em aula2/')
print(resultados_md)

RESULTADOS.md criado em aula2/
# Aula Prática 2 — Resultados

## 2.1 — Caracterização do Dataset

| Classe | N.º imagens | % | Dimensão típica | Observações |
|--------|-------------|---|-----------------|-------------|
| Parasitized | 13779 | 50.0% | 133×133 | dimensões variáveis (range 348×345) |
| Uninfected | 13779 | 50.0% | 130×130 | dimensões variáveis (range 198×186) |

**Total:** 27558 imagens  
**Grupos únicos:** 200  
**Imagens corrompidas:** 0  
**Duplicados (md5):** 0 imagens em 0 grupos

## 2.2 — Divisão sem fugas

**Método:** Reutilização dos splits do D1 (GroupShuffleSplit, seed 42).  
**Justificação:** Garante consistência com o trabalho anterior e respeita a integridade de grupos.

| Verificação | Resultado |
|------------|----------|
| Interseção MD5 train∩val | 0 |
| Interseção MD5 train∩test | 0 |
| Interseção MD5 val∩test | 0 |
| Interseção grupo train∩val | 0 |
| Interseção grupo train∩test | 0 |
| Interseção grupo val∩test | 0 |

| Split | N | Parasitized % | Uni

In [22]:
# ─── Atualizar AI_LOG.md ──────────────────────────────────────────────
import datetime

ai_log_path = ROOT / 'AI_LOG.md'
today = datetime.date.today().strftime('%Y-%m-%d')

new_entry = (
    f'| {today} | Antigravity (Claude Opus 4.6) | '
    f'Aula Prática 2 completa — caracterização do dataset (2.1), '
    f'divisão sem fugas por grupo e md5 (2.2), '
    f'baseline CNN 4×Conv+GAP 30 épocas (2.3), '
    f'curvas de aprendizagem e diagnóstico (2.4) | '
    f'Aceite: notebook aula2/aula2_P1.ipynb com todas as secções, '
    f'reutilização dos splits do D1 por consistência, '
    f'arquitetura exata do enunciado, '
    f'RESULTADOS.md com tabelas preenchidas. '
    f'Tempo de treino: {train_time:.0f}s, '
    f'melhor época: {best_epoch}, '
    f'acc val: {best_val_acc:.4f}. | |\n'
)

with open(ai_log_path, 'r', encoding='utf-8') as f:
    content = f.read()

# Adicionar no fim
content = content.rstrip() + '\n' + new_entry

with open(ai_log_path, 'w', encoding='utf-8') as f:
    f.write(content)

print(f'AI_LOG.md atualizado com entrada de {today}')
print('Campo de verificação deixado em branco para preenchimento manual.')

AI_LOG.md atualizado com entrada de 2026-10-03
Campo de verificação deixado em branco para preenchimento manual.


In [23]:
# ─── Resumo final ─────────────────────────────────────────────────────
print('=' * 60)
print('RESUMO FINAL — Aula Prática 2')
print('=' * 60)
print(f'Tempo de treino:            {train_time:.1f}s ({train_time/60:.1f} min)')
print(f'Melhor época (min val_loss): {best_epoch}')
print(f'Acc treino (melhor época):  {best_train_acc:.4f} ({best_train_acc*100:.2f}%)')
print(f'Acc val (melhor época):     {best_val_acc:.4f} ({best_val_acc*100:.2f}%)')
print(f'Classificador maioritário:  {majority_acc:.4f} ({majority_acc*100:.2f}%)')
print(f'Diagnóstico:                {diagnostico}')
print(f'\nFicheiros gerados:')
print(f'  aula2/aula2_P1.ipynb      (este notebook)')
print(f'  aula2/splits.csv           ({len(splits_final)} linhas)')
print(f'  aula2/history_baseline.csv ({EPOCHS} épocas)')
print(f'  aula2/curvas_baseline.png')
print(f'  aula2/RESULTADOS.md')
print(f'  AI_LOG.md (atualizado)')
print(f'  .gitignore (aula2/split/ adicionado)')
print(f'\nAssunções:')
print(f'  - Reutilizados os splits do D1 (GroupShuffleSplit, seed 42)')
print(f'  - Dataset em data/raw/cell_images/ (2 classes)')
print(f'  - Backend Keras 3 + PyTorch')
print(f'  - Conjunto de teste NÃO avaliado')
print('=' * 60)

RESUMO FINAL — Aula Prática 2
Tempo de treino:            847.0s (14.1 min)
Melhor época (min val_loss): 7
Acc treino (melhor época):  0.9491 (94.91%)
Acc val (melhor época):     0.9627 (96.27%)
Classificador maioritário:  0.5004 (50.04%)
Diagnóstico:                Adequado

Ficheiros gerados:
  aula2/aula2_P1.ipynb      (este notebook)
  aula2/splits.csv           (27558 linhas)
  aula2/history_baseline.csv (30 épocas)
  aula2/curvas_baseline.png
  aula2/RESULTADOS.md
  AI_LOG.md (atualizado)
  .gitignore (aula2/split/ adicionado)

Assunções:
  - Reutilizados os splits do D1 (GroupShuffleSplit, seed 42)
  - Dataset em data/raw/cell_images/ (2 classes)
  - Backend Keras 3 + PyTorch
  - Conjunto de teste NÃO avaliado
